# 02 Tokenization Behavior Lab

This notebook compares how three open-source tokenizers split different kinds of text.

Goal:
- understand why token counts change across models
- measure token count, tokens per word, and tokens per character
- save a CSV report and four matplotlib graphs

## What We Will Study

We are using only tokenizers in this step. We are not loading full models, generating text, or comparing against the OpenAI API yet.

The notebook uses these tokenizer targets:
- `Qwen/Qwen2.5-1.5B-Instruct`
- `mistralai/Mistral-7B-Instruct-v0.2`
- `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

If a tokenizer is gated or unavailable, the Python utility falls back to an easier public tokenizer and prints the reason.

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from tokenizer_utils import (
    CASES_PATH,
    FIGURES_DIR,
    REPORT_PATH,
    build_summary,
    create_graphs,
    load_all_tokenizers,
    load_cases,
    analyze_multiple_texts,
    save_report,
    REQUESTED_MODEL_NAMES,
)

print("Project root:", PROJECT_ROOT)
print("Cases file:", CASES_PATH)
print("Report path:", REPORT_PATH)
print("Figures directory:", FIGURES_DIR)

Project root: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference
Cases file: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/examples/tokenization_cases.json
Report path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/tokenization_behavior_report.csv
Figures directory: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/figures


## Load the Cases

The JSON file stores one category and one text example per case. This keeps the experiment organized and easy to extend later.

In [9]:
cases = load_cases(CASES_PATH)
cases

[{'category': 'simple English sentence',
  'text': 'Hello, I am learning how tokenizers break text into model-readable pieces.'},
 {'category': 'long English paragraph',
  'text': 'Large language model inference starts long before text generation. A tokenizer first converts raw text into smaller vocabulary units, and those token choices affect cost, context usage, and latency. Even when two models read the same sentence, their tokenizers may split code, punctuation, and multilingual text differently, which changes how much work the model needs to do.'},
 {'category': 'Python code',
  'text': 'def estimate_cost(tokens, price_per_million=0.15):\n    total = (tokens / 1_000_000) * price_per_million\n    return round(total, 6)'},
 {'category': 'JSON payload',
  'text': '{"model":"qwen2.5","prompt_tokens":128,"decode_tokens":64,"use_cache":true,"metadata":{"team":"infra","env":"dev"}}'},
 {'category': 'SQL query',
  'text': "SELECT model_name, AVG(latency_ms) AS avg_latency FROM inference_l

## Load the Tokenizers

This step downloads or reuses cached tokenizer files only. No full model weights are loaded.

In [10]:
tokenizers = load_all_tokenizers(REQUESTED_MODEL_NAMES)
list(tokenizers.keys())

Could not load 'google/gemma-2-2b-it' because of: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b-it.
401 Client Error. (Request ID: Root=1-6a3b029a-2d46ae2269c11ecd0a93e550;97a303b1-3448-47dd-89bf-7e252ba13288)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in.
Falling back to 'gpt2' for this experiment.


['Qwen/Qwen2.5-1.5B-Instruct',
 'mistralai/Mistral-7B-Instruct-v0.2',
 'google/gemma-2-2b-it -> gpt2']

## Run the Analysis

For every text case and every tokenizer, we measure:
- character count
- word count
- token count
- tokens per word
- tokens per character
- token pieces

In [11]:
results = analyze_multiple_texts(cases, tokenizers)
df = save_report(results, REPORT_PATH)
df.head(10)

,category,tokenizer_name,text,character_count,word_count,token_count,tokens_per_word,tokens_per_character,token_pieces
26,Telugu sentence,google/gemma-2-2b-it -> gpt2,నేను ఎల్ ఎల్ ఎం ఇన్‌ఫరెన్స్ గురించి నేర్చుకుంట...,95,10,261,26.1000,2.7474,"[à, °, ¨, à, ±, ĩ, à, °, ¨, à, ±, ģ, Ġ, à, °, ..."
25,Telugu sentence,mistralai/Mistral-7B-Instruct-v0.2,నేను ఎల్ ఎల్ ఎం ఇన్‌ఫరెన్స్ గురించి నేర్చుకుంట...,95,10,150,15.0000,1.5789,"[▁, న, <0xE0>, <0xB1>, <0x87>, న, ు, ▁, <0xE0>..."
24,Telugu sentence,Qwen/Qwen2.5-1.5B-Instruct,నేను ఎల్ ఎల్ ఎం ఇన్‌ఫరెన్స్ గురించి నేర్చుకుంట...,95,10,142,14.2000,1.4947,"[à°¨, à±, ĩ, à°¨, à±, ģ, Ġà°, İ, à°², à±, į, Ġ..."
16,numbers-heavy text,mistralai/Mistral-7B-Instruct-v0.2,"Batch sizes 1, 4, 8, 16, 32 and 64 produced la...",122,21,82,3.9048,0.6721,"[▁B, atch, ▁sizes, ▁, 1, ,, ▁, 4, ,, ▁, 8, ,, ..."
15,numbers-heavy text,Qwen/Qwen2.5-1.5B-Instruct,"Batch sizes 1, 4, 8, 16, 32 and 64 produced la...",122,21,81,3.8571,0.6639,"[Batch, Ġsizes, Ġ, 1, ,, Ġ, 4, ,, Ġ, 8, ,, Ġ, ..."
4,long English paragraph,mistralai/Mistral-7B-Instruct-v0.2,Large language model inference starts long bef...,375,57,76,1.3333,0.2027,"[▁Large, ▁language, ▁model, ▁in, ference, ▁sta..."
5,long English paragraph,google/gemma-2-2b-it -> gpt2,Large language model inference starts long bef...,375,57,71,1.2456,0.1893,"[Large, Ġlanguage, Ġmodel, Ġinference, Ġstarts..."
3,long English paragraph,Qwen/Qwen2.5-1.5B-Instruct,Large language model inference starts long bef...,375,57,69,1.2105,0.1840,"[Large, Ġlanguage, Ġmodel, Ġinference, Ġstarts..."
17,numbers-heavy text,google/gemma-2-2b-it -> gpt2,"Batch sizes 1, 4, 8, 16, 32 and 64 produced la...",122,21,55,2.6190,0.4508,"[B, atch, Ġsizes, Ġ1, ,, Ġ4, ,, Ġ8, ,, Ġ16, ,,..."
7,Python code,mistralai/Mistral-7B-Instruct-v0.2,"def estimate_cost(tokens, price_per_million=0....",130,13,54,4.1538,0.4154,"[▁def, ▁estimate, _, cost, (, tokens, ,, ▁pric..."


## Full Report Sorted by Token Count

The report is sorted from highest token count to lowest token count so we can immediately spot which text examples are most expensive for the tokenizer to represent.

In [12]:
df

,category,tokenizer_name,text,character_count,word_count,token_count,tokens_per_word,tokens_per_character,token_pieces
26,Telugu sentence,google/gemma-2-2b-it -> gpt2,నేను ఎల్ ఎల్ ఎం ఇన్‌ఫరెన్స్ గురించి నేర్చుకుంట...,95,10,261,26.1000,2.7474,"[à, °, ¨, à, ±, ĩ, à, °, ¨, à, ±, ģ, Ġ, à, °, ..."
25,Telugu sentence,mistralai/Mistral-7B-Instruct-v0.2,నేను ఎల్ ఎల్ ఎం ఇన్‌ఫరెన్స్ గురించి నేర్చుకుంట...,95,10,150,15.0000,1.5789,"[▁, న, <0xE0>, <0xB1>, <0x87>, న, ు, ▁, <0xE0>..."
24,Telugu sentence,Qwen/Qwen2.5-1.5B-Instruct,నేను ఎల్ ఎల్ ఎం ఇన్‌ఫరెన్స్ గురించి నేర్చుకుంట...,95,10,142,14.2000,1.4947,"[à°¨, à±, ĩ, à°¨, à±, ģ, Ġà°, İ, à°², à±, į, Ġ..."
16,numbers-heavy text,mistralai/Mistral-7B-Instruct-v0.2,"Batch sizes 1, 4, 8, 16, 32 and 64 produced la...",122,21,82,3.9048,0.6721,"[▁B, atch, ▁sizes, ▁, 1, ,, ▁, 4, ,, ▁, 8, ,, ..."
15,numbers-heavy text,Qwen/Qwen2.5-1.5B-Instruct,"Batch sizes 1, 4, 8, 16, 32 and 64 produced la...",122,21,81,3.8571,0.6639,"[Batch, Ġsizes, Ġ, 1, ,, Ġ, 4, ,, Ġ, 8, ,, Ġ, ..."
4,long English paragraph,mistralai/Mistral-7B-Instruct-v0.2,Large language model inference starts long bef...,375,57,76,1.3333,0.2027,"[▁Large, ▁language, ▁model, ▁in, ference, ▁sta..."
5,long English paragraph,google/gemma-2-2b-it -> gpt2,Large language model inference starts long bef...,375,57,71,1.2456,0.1893,"[Large, Ġlanguage, Ġmodel, Ġinference, Ġstarts..."
3,long English paragraph,Qwen/Qwen2.5-1.5B-Instruct,Large language model inference starts long bef...,375,57,69,1.2105,0.1840,"[Large, Ġlanguage, Ġmodel, Ġinference, Ġstarts..."
17,numbers-heavy text,google/gemma-2-2b-it -> gpt2,"Batch sizes 1, 4, 8, 16, 32 and 64 produced la...",122,21,55,2.6190,0.4508,"[B, atch, Ġsizes, Ġ1, ,, Ġ4, ,, Ġ8, ,, Ġ16, ,,..."
7,Python code,mistralai/Mistral-7B-Instruct-v0.2,"def estimate_cost(tokens, price_per_million=0....",130,13,54,4.1538,0.4154,"[▁def, ▁estimate, _, cost, (, tokens, ,, ▁pric..."


## Summary Findings

These summary values answer the main Step 2 questions:
- which category is most token-heavy
- which tokenizer has the highest average token count
- which tokenizer has the lowest average token count
- which category varies the most across tokenizers

In [13]:
summary = build_summary(df)
summary

{'most_token_heavy_category': 'Telugu sentence',
 'highest_average_tokenizer': 'google/gemma-2-2b-it -> gpt2',
 'lowest_average_tokenizer': 'Qwen/Qwen2.5-1.5B-Instruct',
 'most_variable_category': 'Telugu sentence'}

## Create the Graphs

We now generate four graphs and save them into `outputs/figures/`.

In [14]:
create_graphs(df, FIGURES_DIR)
sorted(path.name for path in FIGURES_DIR.glob('*.png'))

['character_count_vs_token_count.png',
 'context_usage_by_prompt.png',
 'context_used_percent_by_tokenizer.png',
 'fits_vs_exceeds_context.png',
 'included_vs_dropped_sections.png',
 'packed_prompt_usage.png',
 'remaining_tokens_by_prompt.png',
 'strategy_comparison.png',
 'token_budget_by_section.png',
 'token_count_by_category_and_tokenizer.png',
 'tokenizer_comparison_summary.png',
 'tokens_per_word_by_category_and_tokenizer.png']

## What the Graphs Show

1. **Token count by category and tokenizer**
   This bar chart shows the raw token count for each text category. It helps you see which kinds of text are the most expensive in token terms.

2. **Tokens per word by category and tokenizer**
   This chart normalizes token count by word count. It is useful because two examples may have similar token counts but very different numbers of words.

3. **Character count vs token count**
   This scatter plot shows whether more characters usually lead to more tokens, and whether some tokenizers are more efficient than others for the same amount of text.

4. **Average token count per tokenizer**
   This summary chart compares the average token count produced by each tokenizer across all categories. It gives a quick overall efficiency comparison.

## Why Token Counts Differ Across Tokenizers

Different models learn different vocabularies and merge rules. That means the same text may be stored as one token in one tokenizer, but split into several smaller pieces in another.

These categories often behave differently:
- **Code** because symbols, indentation, and variable names are not normal plain English.
- **JSON** because braces, quotes, colons, and compact formatting create many short pieces.
- **Punctuation-heavy text** because many punctuation marks become separate tokens.
- **Multilingual text** because scripts such as Telugu or Hindi may not map to the same frequent subword units as English.
- **Emoji-heavy text** because emojis are unusual characters and may become separate or multi-part tokens.
- **Whitespace-heavy text** because extra spaces, tabs, and newlines can change token boundaries.

## Step 2 Takeaway

In this step, you learned that token count depends on the tokenizer, not just the sentence itself. That matters because token count affects inference cost, latency, and context usage.

In [2]:
import pandas as pd

token_df = pd.read_csv("../outputs/tokenization_behavior_report.csv")
token_df.head()

,category,tokenizer_name,text,character_count,word_count,token_count,tokens_per_word,tokens_per_character,token_pieces
0,Telugu sentence,google/gemma-2-2b-it -> gpt2,నేను ఎల్ ఎల్ ఎం ఇన్‌ఫరెన్స్ గురించి నేర్చుకుంట...,95,10,261,26.1000,2.7474,"['à', '°', '¨', 'à', '±', 'ĩ', 'à', '°', '¨', ..."
1,Telugu sentence,mistralai/Mistral-7B-Instruct-v0.2,నేను ఎల్ ఎల్ ఎం ఇన్‌ఫరెన్స్ గురించి నేర్చుకుంట...,95,10,150,15.0000,1.5789,"['▁', 'న', '<0xE0>', '<0xB1>', '<0x87>', 'న', ..."
2,Telugu sentence,Qwen/Qwen2.5-1.5B-Instruct,నేను ఎల్ ఎల్ ఎం ఇన్‌ఫరెన్స్ గురించి నేర్చుకుంట...,95,10,142,14.2000,1.4947,"['à°¨', 'à±', 'ĩ', 'à°¨', 'à±', 'ģ', 'Ġà°', 'İ..."
3,numbers-heavy text,mistralai/Mistral-7B-Instruct-v0.2,"Batch sizes 1, 4, 8, 16, 32 and 64 produced la...",122,21,82,3.9048,0.6721,"['▁B', 'atch', '▁sizes', '▁', '1', ',', '▁', '..."
4,numbers-heavy text,Qwen/Qwen2.5-1.5B-Instruct,"Batch sizes 1, 4, 8, 16, 32 and 64 produced la...",122,21,81,3.8571,0.6639,"['Batch', 'Ġsizes', 'Ġ', '1', ',', 'Ġ', '4', '..."


In [3]:
token_screenshot_df = token_df[
    [
        "category",
        "tokenizer_name",
        "character_count",
        "word_count",
        "token_count",
        "tokens_per_word",
        "tokens_per_character"
    ]
]

token_screenshot_df.head(15)

,category,tokenizer_name,character_count,word_count,token_count,tokens_per_word,tokens_per_character
0,Telugu sentence,google/gemma-2-2b-it -> gpt2,95,10,261,26.1000,2.7474
1,Telugu sentence,mistralai/Mistral-7B-Instruct-v0.2,95,10,150,15.0000,1.5789
2,Telugu sentence,Qwen/Qwen2.5-1.5B-Instruct,95,10,142,14.2000,1.4947
3,numbers-heavy text,mistralai/Mistral-7B-Instruct-v0.2,122,21,82,3.9048,0.6721
4,numbers-heavy text,Qwen/Qwen2.5-1.5B-Instruct,122,21,81,3.8571,0.6639
5,long English paragraph,mistralai/Mistral-7B-Instruct-v0.2,375,57,76,1.3333,0.2027
6,long English paragraph,google/gemma-2-2b-it -> gpt2,375,57,71,1.2456,0.1893
7,long English paragraph,Qwen/Qwen2.5-1.5B-Instruct,375,57,69,1.2105,0.1840
8,numbers-heavy text,google/gemma-2-2b-it -> gpt2,122,21,55,2.6190,0.4508
9,Python code,mistralai/Mistral-7B-Instruct-v0.2,130,13,54,4.1538,0.4154


In [4]:
token_screenshot_df = token_screenshot_df.sort_values(
    by="token_count",
    ascending=False
)

token_screenshot_df.head(15)

,category,tokenizer_name,character_count,word_count,token_count,tokens_per_word,tokens_per_character
0,Telugu sentence,google/gemma-2-2b-it -> gpt2,95,10,261,26.1000,2.7474
1,Telugu sentence,mistralai/Mistral-7B-Instruct-v0.2,95,10,150,15.0000,1.5789
2,Telugu sentence,Qwen/Qwen2.5-1.5B-Instruct,95,10,142,14.2000,1.4947
3,numbers-heavy text,mistralai/Mistral-7B-Instruct-v0.2,122,21,82,3.9048,0.6721
4,numbers-heavy text,Qwen/Qwen2.5-1.5B-Instruct,122,21,81,3.8571,0.6639
5,long English paragraph,mistralai/Mistral-7B-Instruct-v0.2,375,57,76,1.3333,0.2027
6,long English paragraph,google/gemma-2-2b-it -> gpt2,375,57,71,1.2456,0.1893
7,long English paragraph,Qwen/Qwen2.5-1.5B-Instruct,375,57,69,1.2105,0.1840
8,numbers-heavy text,google/gemma-2-2b-it -> gpt2,122,21,55,2.6190,0.4508
9,Python code,mistralai/Mistral-7B-Instruct-v0.2,130,13,54,4.1538,0.4154


In [5]:
token_count_comparison = token_df.pivot_table(
    index="category",
    columns="tokenizer_name",
    values="token_count",
    aggfunc="mean"
).reset_index()

token_count_comparison

tokenizer_name,category,Qwen/Qwen2.5-1.5B-Instruct,google/gemma-2-2b-it -> gpt2,mistralai/Mistral-7B-Instruct-v0.2
0,JSON payload,37.0,45.0,44.0
1,Python code,46.0,53.0,54.0
2,SQL query,31.0,41.0,49.0
3,Telugu sentence,142.0,261.0,150.0
4,emoji-heavy text,33.0,41.0,42.0
5,extra spaces and newlines,22.0,28.0,29.0
6,long English paragraph,69.0,71.0,76.0
7,numbers-heavy text,81.0,55.0,82.0
8,punctuation-heavy text,34.0,38.0,41.0
9,simple English sentence,15.0,16.0,17.0


In [6]:
tokens_per_word_comparison = token_df.pivot_table(
    index="category",
    columns="tokenizer_name",
    values="tokens_per_word",
    aggfunc="mean"
).reset_index()

tokens_per_word_comparison.columns.name = None
tokens_per_word_comparison

,category,Qwen/Qwen2.5-1.5B-Instruct,google/gemma-2-2b-it -> gpt2,mistralai/Mistral-7B-Instruct-v0.2
0,JSON payload,37.0000,45.0000,44.0000
1,Python code,3.5385,4.0769,4.1538
2,SQL query,1.7222,2.2778,2.7222
3,Telugu sentence,14.2000,26.1000,15.0000
4,emoji-heavy text,1.4348,1.7826,1.8261
5,extra spaces and newlines,1.2941,1.6471,1.7059
6,long English paragraph,1.2105,1.2456,1.3333
7,numbers-heavy text,3.8571,2.6190,3.9048
8,punctuation-heavy text,1.4783,1.6522,1.7826
9,simple English sentence,1.3636,1.4545,1.5455


In [8]:
context_df = pd.read_csv("../outputs/context_window_report.csv")

context_df.head()

,prompt_name,tokenizer_name,character_count,word_count,prompt_token_count,reserved_output_tokens,context_window,total_required_tokens,remaining_tokens,context_used_percent,fits_context_window
0,multilingual prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,153,21,163,1024,4096,1187,2909,28.9795,True
1,RAG-style prompt with multiple chunks,TinyLlama/TinyLlama-1.1B-Chat-v1.0,600,89,144,1024,4096,1168,2928,28.5156,True
2,RAG-style prompt with multiple chunks,mistralai/Mistral-7B-Instruct-v0.2,600,89,139,1024,4096,1163,2933,28.3936,True
3,multilingual prompt,mistralai/Mistral-7B-Instruct-v0.2,153,21,124,1024,4096,1148,2948,28.0273,True
4,RAG-style prompt with multiple chunks,Qwen/Qwen2.5-1.5B-Instruct,600,89,119,1024,4096,1143,2953,27.9053,True


In [9]:
context_screenshot_df = context_df[
    [
        "prompt_name",
        "tokenizer_name",
        "prompt_token_count",
        "reserved_output_tokens",
        "context_window",
        "total_required_tokens",
        "remaining_tokens",
        "context_used_percent",
        "fits_context_window"
    ]
]

context_screenshot_df.head(15)

,prompt_name,tokenizer_name,prompt_token_count,reserved_output_tokens,context_window,total_required_tokens,remaining_tokens,context_used_percent,fits_context_window
0,multilingual prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,163,1024,4096,1187,2909,28.9795,True
1,RAG-style prompt with multiple chunks,TinyLlama/TinyLlama-1.1B-Chat-v1.0,144,1024,4096,1168,2928,28.5156,True
2,RAG-style prompt with multiple chunks,mistralai/Mistral-7B-Instruct-v0.2,139,1024,4096,1163,2933,28.3936,True
3,multilingual prompt,mistralai/Mistral-7B-Instruct-v0.2,124,1024,4096,1148,2948,28.0273,True
4,RAG-style prompt with multiple chunks,Qwen/Qwen2.5-1.5B-Instruct,119,1024,4096,1143,2953,27.9053,True
5,multilingual prompt,Qwen/Qwen2.5-1.5B-Instruct,109,1024,4096,1133,2963,27.6611,True
6,JSON/tool-calling prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,108,1024,4096,1132,2964,27.6367,True
7,JSON/tool-calling prompt,mistralai/Mistral-7B-Instruct-v0.2,108,1024,4096,1132,2964,27.6367,True
8,long essay prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,92,1024,4096,1116,2980,27.2461,True
9,JSON/tool-calling prompt,Qwen/Qwen2.5-1.5B-Instruct,90,1024,4096,1114,2982,27.1973,True


In [10]:
context_screenshot_df = context_screenshot_df.sort_values(
    by="context_used_percent",
    ascending=False
)

context_screenshot_df.head(15)

,prompt_name,tokenizer_name,prompt_token_count,reserved_output_tokens,context_window,total_required_tokens,remaining_tokens,context_used_percent,fits_context_window
0,multilingual prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,163,1024,4096,1187,2909,28.9795,True
1,RAG-style prompt with multiple chunks,TinyLlama/TinyLlama-1.1B-Chat-v1.0,144,1024,4096,1168,2928,28.5156,True
2,RAG-style prompt with multiple chunks,mistralai/Mistral-7B-Instruct-v0.2,139,1024,4096,1163,2933,28.3936,True
3,multilingual prompt,mistralai/Mistral-7B-Instruct-v0.2,124,1024,4096,1148,2948,28.0273,True
4,RAG-style prompt with multiple chunks,Qwen/Qwen2.5-1.5B-Instruct,119,1024,4096,1143,2953,27.9053,True
5,multilingual prompt,Qwen/Qwen2.5-1.5B-Instruct,109,1024,4096,1133,2963,27.6611,True
6,JSON/tool-calling prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,108,1024,4096,1132,2964,27.6367,True
7,JSON/tool-calling prompt,mistralai/Mistral-7B-Instruct-v0.2,108,1024,4096,1132,2964,27.6367,True
8,long essay prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,92,1024,4096,1116,2980,27.2461,True
9,JSON/tool-calling prompt,Qwen/Qwen2.5-1.5B-Instruct,90,1024,4096,1114,2982,27.1973,True


In [11]:
context_usage_comparison = context_df.pivot_table(
    index="prompt_name",
    columns="tokenizer_name",
    values="context_used_percent",
    aggfunc="mean"
).reset_index()

context_usage_comparison.columns.name = None
context_usage_comparison

,prompt_name,Qwen/Qwen2.5-1.5B-Instruct,TinyLlama/TinyLlama-1.1B-Chat-v1.0,mistralai/Mistral-7B-Instruct-v0.2
0,JSON/tool-calling prompt,9.089500,9.327533,9.327533
1,RAG-style prompt with multiple chunks,9.473000,9.803600,9.737500
2,chat history prompt,8.891133,9.023367,9.010144
3,code debugging prompt,8.851444,8.957244,8.957244
4,long essay prompt,8.996933,9.115933,9.089500
5,multilingual prompt,9.340744,10.054844,9.539111
6,short user question,8.124100,8.124100,8.137344


In [12]:
fits_summary = context_df["fits_context_window"].value_counts().reset_index()

fits_summary.columns = ["fits_context_window", "count"]

fits_summary

,fits_context_window,count
0,True,189


In [13]:
fits_summary["percentage"] = (
    fits_summary["count"] / fits_summary["count"].sum() * 100
).round(2)

fits_summary

,fits_context_window,count,percentage
0,True,189,100.0


In [14]:
token_df[
    [
        "category",
        "tokenizer_name",
        "word_count",
        "token_count",
        "tokens_per_word"
    ]
].sort_values("token_count", ascending=False).head(12)

,category,tokenizer_name,word_count,token_count,tokens_per_word
0,Telugu sentence,google/gemma-2-2b-it -> gpt2,10,261,26.1000
1,Telugu sentence,mistralai/Mistral-7B-Instruct-v0.2,10,150,15.0000
2,Telugu sentence,Qwen/Qwen2.5-1.5B-Instruct,10,142,14.2000
3,numbers-heavy text,mistralai/Mistral-7B-Instruct-v0.2,21,82,3.9048
4,numbers-heavy text,Qwen/Qwen2.5-1.5B-Instruct,21,81,3.8571
5,long English paragraph,mistralai/Mistral-7B-Instruct-v0.2,57,76,1.3333
6,long English paragraph,google/gemma-2-2b-it -> gpt2,57,71,1.2456
7,long English paragraph,Qwen/Qwen2.5-1.5B-Instruct,57,69,1.2105
8,numbers-heavy text,google/gemma-2-2b-it -> gpt2,21,55,2.6190
9,Python code,mistralai/Mistral-7B-Instruct-v0.2,13,54,4.1538


In [15]:
tokenizer_summary = token_df.groupby("tokenizer_name").agg(
    avg_token_count=("token_count", "mean"),
    max_token_count=("token_count", "max"),
    avg_tokens_per_word=("tokens_per_word", "mean")
).reset_index()

tokenizer_summary

,tokenizer_name,avg_token_count,max_token_count,avg_tokens_per_word
0,Qwen/Qwen2.5-1.5B-Instruct,51.0,142,6.70991
1,google/gemma-2-2b-it -> gpt2,64.9,261,8.78557
2,mistralai/Mistral-7B-Instruct-v0.2,58.4,150,7.79742


In [16]:
context_df[
    [
        "prompt_name",
        "tokenizer_name",
        "prompt_token_count",
        "reserved_output_tokens",
        "context_window",
        "remaining_tokens",
        "context_used_percent",
        "fits_context_window"
    ]
].sort_values("context_used_percent", ascending=False).head(12)

,prompt_name,tokenizer_name,prompt_token_count,reserved_output_tokens,context_window,remaining_tokens,context_used_percent,fits_context_window
0,multilingual prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,163,1024,4096,2909,28.9795,True
1,RAG-style prompt with multiple chunks,TinyLlama/TinyLlama-1.1B-Chat-v1.0,144,1024,4096,2928,28.5156,True
2,RAG-style prompt with multiple chunks,mistralai/Mistral-7B-Instruct-v0.2,139,1024,4096,2933,28.3936,True
3,multilingual prompt,mistralai/Mistral-7B-Instruct-v0.2,124,1024,4096,2948,28.0273,True
4,RAG-style prompt with multiple chunks,Qwen/Qwen2.5-1.5B-Instruct,119,1024,4096,2953,27.9053,True
5,multilingual prompt,Qwen/Qwen2.5-1.5B-Instruct,109,1024,4096,2963,27.6611,True
6,JSON/tool-calling prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,108,1024,4096,2964,27.6367,True
7,JSON/tool-calling prompt,mistralai/Mistral-7B-Instruct-v0.2,108,1024,4096,2964,27.6367,True
8,long essay prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,92,1024,4096,2980,27.2461,True
9,JSON/tool-calling prompt,Qwen/Qwen2.5-1.5B-Instruct,90,1024,4096,2982,27.1973,True


In [17]:
context_window_summary = context_df.groupby("context_window").agg(
    avg_context_used_percent=("context_used_percent", "mean"),
    min_remaining_tokens=("remaining_tokens", "min"),
    total_cases=("fits_context_window", "count")
).reset_index()

context_window_summary

,context_window,avg_context_used_percent,min_remaining_tokens,total_cases
0,4096,16.788738,2909,63
1,8192,8.394367,7005,63
2,32768,2.098589,31581,63
